In [ ]:
from core.hydrodispatchenv import HydroDispatchEnv

import numpy as np
import matplotlib.pyplot as plt

env = HydroDispatchEnv(inflow_m3s=25.0)

# strategia uno : constant discharge
obs, info = env.reset(seed=42)
constant_rewards = []
for _ in range(env.HOURS_PER_EPISODE):
    obs, reward, term, trunc, info = env.step(np.array([25.0]))
    constant_rewards.append(reward)
    if term:
        break
total_constant = sum(constant_rewards)

# strategia dos: peak-only discharge
obs, info = env.reset(seed=42)
peak_rewards = []
for step in range(env.HOURS_PER_EPISODE):
    hour = step % 24
    # Generate at max during peak (10-16) and shoulder (16-22) to avoid spill, hold otherwise
    if 10 <= hour < 16:
        # Peak (6 hours): Full capacity
        action = np.array([env.TURBINE_Q_MAX])
    elif 16 <= hour < 22:
        # Shoulder (6 hours): Keep running full capacity to avoid spill
        action = np.array([env.TURBINE_Q_MAX])
    else:
        # Hold water
        action = np.array([0.0])
    obs, reward, term, trunc, info = env.step(action)
    peak_rewards.append(reward)
    if term: break
total_peak = sum(peak_rewards)

print("=== ECONOMIC COMPARISON ===")
print(f"Strategy 1 (Constant 25 m³/s): ${total_constant:,.0f}")
print(f"Strategy 2 (Peak+Shoulder):    ${total_peak:,.0f}")
print(f"Advantage: {(total_peak/total_constant - 1)*100:.1f}% more revenue")

# Plot hourly rewards comparison
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

hours = range(min(48, len(constant_rewards)))  # Show first 2 days
axes[0].bar(hours, constant_rewards[:48], alpha=0.7, color='gray',
            label='Constant Dispatch')
axes[0].bar(hours, peak_rewards[:48], alpha=0.5, color='green',
            label='Peak+Shoulder Dispatch')
axes[0].set_ylabel("Revenue ($)")
axes[0].set_title("Hourly Revenue: First 48 Hours")
axes[0].legend()

# Tariff overlay
tariffs = [env._get_tariff(h % 24) for h in range(48)]
axes[1].step(hours, tariffs, 'r-', linewidth=2, where='mid')
axes[1].fill_between(hours, tariffs, alpha=0.2, color='red', step='mid')
axes[1].set_ylabel("Tariff ($/MWh)")
axes[1].set_xlabel("Hour")
axes[1].set_title("Time-of-Day Tariff Schedule")

plt.tight_layout()
plt.show()